# fact_pagos
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Gold (nivel 2) · hecho Pagos |
| Grano | un pago (estado actual) con su recorrido de estados. |
| Fuentes | silver.pagos + silver.pagos_historial_estados + silver.pedidos · dim_clientes |
| Métricas | n_cambios_estado, fecha_aprobacion, minutos_hasta_aprobacion y n_intentos_pago (pagos del mismo pedido: rechazados y reintentos). |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import Window
from pyspark.sql import functions as F

from comun_gold import cliente_vigente_en, leer_silver, sk_fecha

## 2. Tabla `fact_pagos`
Pagos (estado actual) con su recorrido de estados: cambios, tiempo hasta aprobarse e intentos del pedido.

In [ ]:
@dlt.table(name="fact_pagos", cluster_by=["sk_fecha"],
           comment="Pagos (estado actual) con su recorrido de estados: cambios, tiempo hasta aprobarse e intentos del pedido.")
def fact_pagos():
    pagos = leer_silver("pagos")
    historial = (leer_silver("pagos_historial_estados").groupBy("id_pago")
                 .agg((F.count("*") - 1).alias("n_cambios_estado"),
                      F.min(F.when(F.col("estado") == "aprobado", F.col("__START_AT._fecha_efectiva")))
                       .alias("fecha_aprobacion")))
    pedidos = leer_silver("pedidos").select("id_pedido", "id_cliente", "fecha_pedido")
    intentos = Window.partitionBy("id_pedido")
    df = (pagos.join(historial, "id_pago", "left").join(pedidos, "id_pedido", "left")
          .withColumn("n_intentos_pago", F.count("*").over(intentos)))
    df = cliente_vigente_en(df, dlt.read("dim_clientes"), "id_cliente", "fecha_pedido")
    return df.select(
        sk_fecha("fecha_pago").alias("sk_fecha"), "sk_cliente", "id_pago", "id_pedido", "id_cliente",
        "metodo_pago", "monto", "estado", "fecha_pago", "fecha_aprobacion",
        F.round((F.unix_timestamp("fecha_aprobacion") - F.unix_timestamp("fecha_pago")) / 60, 1)
         .alias("minutos_hasta_aprobacion"),
        F.coalesce("n_cambios_estado", F.lit(0)).alias("n_cambios_estado"), "n_intentos_pago")